# Notebook 05: Unsupervised Learning and Assignment 3 Walkthrough

This session is split into two halves.

- Part 1 introduces unsupervised learning with K-means clustering.
- Part 2 walks through the harder Assignment 3 data-analysis tasks using the motorcycle dataset.

The exercises are a little more challenging than the supervised learning practice. You will write helper functions, compare model settings, and justify choices using evidence.


## Learning Goals

By the end of this session, you should be able to:

- Explain how unsupervised learning differs from supervised learning.
- Explain why scaling matters before distance-based clustering.
- Fit K-means with different values of `k` and compare the results.
- Profile clusters using summary statistics instead of guessing names.
- Clean Assignment 3 data before filtering, correlation, plotting, and clustering.
- Interpret Assignment 3 results cautiously and with evidence.


## Warm-up Questions

1. In supervised learning, what does `y` represent?
2. What changes when a dataset has features but no answer labels?
3. Why might two columns with very different units cause problems for distance-based methods?
4. What would make a cluster interpretation convincing rather than just a guess?


# Part 1: Unsupervised Learning


## Section 1: From Prediction to Pattern Finding

Supervised learning asks a model to predict known labels. Unsupervised learning asks a model to find structure when labels are not provided.

K-means clustering creates groups by repeatedly assigning rows to nearby cluster centers. Because it uses distance, feature scale matters.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score


## Section 2: Load Data Without Using Labels

We will use the iris measurements, but pretend the species labels are hidden. This lets us practice clustering first, then check later whether the clusters line up with known species.


In [ ]:
iris = load_iris()
iris_features = pd.DataFrame(iris.data, columns=iris.feature_names)

species_names = pd.Series(iris.target, name="species").map(dict(enumerate(iris.target_names)))

iris_features.head()


### Exercise 2.1: Choose Evidence Before Clustering

Before running K-means, choose two features and make a scatter plot.

Questions:

1. Which two columns did you choose?
2. Do you see one group, several groups, or mostly overlap?
3. What would make this plot misleading?

Try at least two different feature pairs before moving on.


In [ ]:
feature_x = "petal length (cm)"
feature_y = "petal width (cm)"

plt.figure(figsize=(6, 4))
plt.scatter(iris_features[feature_x], iris_features[feature_y])
plt.xlabel(feature_x)
plt.ylabel(feature_y)
plt.title("Iris features before clustering")
plt.show()


## Section 3: Scale Features and Fit a First K-means Model

K-means is affected by distance. Standardizing puts each feature on a comparable scale before clustering.


In [ ]:
scaler = StandardScaler()
iris_scaled = scaler.fit_transform(iris_features)

kmeans_3 = KMeans(n_clusters=3, random_state=0, n_init=10)
iris_cluster_labels = kmeans_3.fit_predict(iris_scaled)

pd.Series(iris_cluster_labels, name="cluster").value_counts().sort_index()


### Exercise 3.1: Compare Several Values of `k`

Complete the loop below for `k = 2`, `3`, `4`, `5`, and `6`.

For each `k`, record:

- `inertia`: lower usually means tighter clusters, but it always decreases as `k` grows.
- `silhouette`: higher usually means better-separated clusters, but it is still only a guide.

This is more challenging than only changing one number because you must compare two metrics and explain the tradeoff.


In [ ]:
k_summaries = []

for k in [2, 3, 4, 5, 6]:
    # TODO: create a KMeans model with this value of k.
    # TODO: fit the model and get labels.
    # TODO: calculate inertia and silhouette_score.
    # TODO: append a dictionary with k, inertia, and silhouette.
    pass

pd.DataFrame(k_summaries)


## Section 4: Profile Clusters Before Naming Them

A cluster number is not an explanation. To interpret clusters, summarize the feature values inside each cluster.


In [ ]:
iris_clustered = iris_features.copy()
iris_clustered["cluster"] = iris_cluster_labels

# TODO: use groupby("cluster").mean() to profile the clusters.
cluster_profile = None

# TODO: only after clustering, add species_names and compare cluster labels with true species.
# Hint: pd.crosstab(...) is useful here.
species_check = None

cluster_profile


### Exercise 4.1: Make a Careful Interpretation

Fill in the dictionary below.

Your answer should include:

- one pattern you see,
- one piece of numerical evidence,
- one caution about what K-means does not prove.


In [ ]:
cluster_notes = {
    "pattern": "",
    "evidence": "",
    "caution": "",
}

cluster_notes


# Part 2: Assignment 3 Walkthrough

Assignment 3 asks you to load a motorcycle dataset, clean it, inspect it, filter it, compute correlations, make plots, and apply K-means.

This walkthrough gives structure and checkpoints, but it is not meant to be a copy-paste answer key. Try to finish each helper function with your own decisions.


## Section 5: Load the Motorcycle Dataset

The course repo stores the Assignment 3 data in `data/all_bikez_curated.csv`. The path helper below works whether Jupyter starts from the repo root or from the notebook folder.


In [ ]:
data_candidates = [
    Path("data/all_bikez_curated.csv"),
    Path("../../data/all_bikez_curated.csv"),
    Path("all_bikez_curated.csv"),
]

DATA_PATH = next((path for path in data_candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("Could not find all_bikez_curated.csv. Check your notebook working directory.")

motorcycle_info_raw = pd.read_csv(DATA_PATH, low_memory=False)

print("Loaded:", DATA_PATH)
print("Raw shape:", motorcycle_info_raw.shape)
motorcycle_info_raw.head()


## Section 6: Cleaning Challenge

The homework needs a clean dataframe and a numeric-only dataframe. Instead of writing one long cell, write a helper function.

Requirements:

1. Make a copy of the raw dataframe.
2. Convert `Stroke (mm)` to numeric if needed.
3. Decide how to handle missing values and explain the cost of that choice.
4. Return both the cleaned dataframe and the numeric-only dataframe.


In [ ]:
def prepare_motorcycle_data(raw_data):
    cleaned = raw_data.copy()

    # TODO: convert "Stroke (mm)" to numeric safely.
    # Hint: remove commas before using pd.to_numeric(..., errors="coerce").

    # TODO: decide how to handle missing values.
    # Hint: compare row counts before and after dropping rows.

    # TODO: select only numeric columns.
    numeric_data = pd.DataFrame()

    return cleaned, numeric_data


motorcycle_info, motorcycle_info_num = prepare_motorcycle_data(motorcycle_info_raw)

print("Raw shape:", motorcycle_info_raw.shape)
print("Cleaned shape:", motorcycle_info.shape)
print("Numeric columns:", list(motorcycle_info_num.columns))
motorcycle_info_num.head()


## Section 7: Filtering and Correlation Challenge

These helper functions connect directly to the homework tests.

Make them reusable: the threshold, target column, and number of features should be parameters, not hard-coded inside the function body.


In [ ]:
def displacement_percentage(data, threshold):
    # TODO: calculate the percentage of rows with Displacement (ccm) <= threshold.
    # Hint: use the mean of a Boolean Series, then multiply by 100.
    return None


def top_correlated_features(numeric_data, target_column, n=3):
    # TODO: compute the correlation matrix.
    # TODO: remove the target column so it does not choose itself.
    # TODO: sort by absolute correlation and return the top n features.
    return pd.Series(dtype=float)


percentage_1198 = displacement_percentage(motorcycle_info, 1198)
top_torque_features = top_correlated_features(motorcycle_info_num, "Torque (Nm)", n=3)

print("Percentage <= 1198:", percentage_1198)
top_torque_features


## Section 8: Multi-condition Filtering and Plots

This is a step up from the supervised learning exercises: combine multiple conditions, inspect the result, and then make a readable plot.


In [ ]:
# TODO: create yamaha_sport with Yamaha motorcycles where:
# - Brand is yamaha,
# - Category is Sport,
# - Displacement (ccm) is greater than 950.
#
# Make the text comparisons robust to capitalization.

yamaha_sport = pd.DataFrame()
yamaha_sport_models = []

yamaha_sport_models


In [ ]:
# TODO: find the top 15 brands by motorcycle count.
# TODO: create a horizontal bar chart.
# TODO: label each bar with the count.

top_15_brands = None

top_15_brands


## Section 9: Assignment 3 K-means Walkthrough

The homework asks for K-means with `k = 3`. The challenge is not only fitting the model. You also need to decide which numeric columns to use, whether to scale them, and how to visualize the clusters.


In [ ]:
cluster_columns = ["Displacement (ccm)", "Torque (Nm)", "Power (hp)", "Dry weight (kg)"]
available_cluster_columns = [col for col in cluster_columns if col in motorcycle_info_num.columns]

print("Available cluster columns:", available_cluster_columns)

if not available_cluster_columns:
    print("Complete the cleaning step before clustering.")
else:
    # TODO: create cluster_data from available_cluster_columns and drop missing rows.
    # TODO: standardize cluster_data.
    # TODO: fit KMeans with n_clusters=3.
    # TODO: attach cluster labels to the matching motorcycle rows.
    # TODO: use PCA with 2 components to make a clustering plot.
    pass


## Reflection Questions

1. Why does K-means need features but not labels?
2. Why is `k = 3` a modeling choice rather than a fact from the data?
3. Which Assignment 3 task was most sensitive to cleaning decisions?
4. What should you say before interpreting a cluster as a real-world motorcycle category?


## Optional Stretch

After finishing the required version, compare two clustering choices:

- K-means on all numeric columns.
- K-means on only `Displacement (ccm)`, `Torque (Nm)`, `Power (hp)`, and `Dry weight (kg)`.

Write two sentences explaining which result is easier to interpret and why.
